In [ ]:
pip install cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 88.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd

import pandas as pd

# Load the replications CSV
replications = pd.read_csv("/content/WorldBank_Replications - WorldBank_Replications.csv")

# Load the main.xlsx file, replace '/path/to/your/main.xlsx' with the correct file path
main_df = pd.read_excel("/content/wbg_Updated_5.xlsx")

def get_unique_replicated_projects(replications_df, main_df):
    # Rename 'Project ID' to match document_id keys
    main_df = main_df.rename(columns={'Project ID': 'document_id', 'Country': 'country'})

    # Merge document_id_1 to get country
    rep_1 = replications_df[['document_id_1']].merge(
        main_df[['document_id', 'country']],
        left_on='document_id_1',
        right_on='document_id',
        how='left'
    )
    rep_1 = rep_1[['document_id_1', 'country']].rename(columns={'document_id_1': 'document_id'})

    # Merge document_id_2 to get country
    rep_2 = replications_df[['document_id_2']].merge(
        main_df[['document_id', 'country']],
        left_on='document_id_2',
        right_on='document_id',
        how='left'
    )
    rep_2 = rep_2[['document_id_2', 'country']].rename(columns={'document_id_2': 'document_id'})

    # Combine both and deduplicate
    replicated_projects = pd.concat([rep_1, rep_2], ignore_index=True)
    replicated_projects = replicated_projects.dropna(subset=['country'])
    replicated_projects = replicated_projects.drop_duplicates()

    # Count unique replicated projects per country
    return replicated_projects.groupby('country')['document_id'].nunique().to_dict()


# Step 1: Count total projects per country
country_project_counts = main_df['Country'].value_counts().to_dict()


print(replications.columns.tolist())

country_unique_replications = get_unique_replicated_projects(replications, main_df)


# Step 3: Compute replication percentages
replication_stats = []
for country, total_projects in country_project_counts.items():
    if total_projects >= 10:  # Only consider countries with at least 10 projects
        replicated_projects = country_unique_replications.get(country, 0)
        percentage = (replicated_projects / total_projects) * 100 if total_projects > 0 else 0
        replication_stats.append({
            'Country': country,
            'Total_Projects': total_projects,
            'Unique_Replicated_Documents': replicated_projects,
            'Replication_Percentage': percentage
        })

replication_stats_df = pd.DataFrame(replication_stats)

# Step 4: Sort and select top 15 countries
replication_stats_df = replication_stats_df.sort_values(by='Replication_Percentage', ascending=False)
top_15_countries = replication_stats_df.head(22)

print("Top 15 Countries by Replication Percentage (minimum 10 projects):")
print(top_15_countries)


['project_id_1', 'document_id_1', 'project_year_1', 'project_id_2', 'document_id_2', 'project_year_2', 'sector', 'distance', 'z_score']
Top 15 Countries by Replication Percentage (minimum 10 projects):
                    Country  Total_Projects  Unique_Replicated_Documents  \
83               Azerbaijan              11                           11   
52  EUROPE AND CENTRAL ASIA              17                           16   
62   Bosnia and Herzegovina              15                           14   
69              Afghanistan              14                           13   
86    EAST ASIA AND PACIFIC              11                           10   
72                  Armenia              13                           11   
75              El Salvador              13                           11   
25               SOUTH ASIA              25                           21   
61                  Romania              16                           13   
35                Sri Lanka           

In [ ]:
top_country_partners = {}

# We'll prepare a lookup for document_id -> country
project_country_lookup = main_df.set_index('Project ID')['Country'].to_dict()

for country in top_15_countries['Country']:
    # Filter rows where source country matches `country`
    replications_with_country_1 = replications[replications['document_id_1'].map(project_country_lookup) == country].copy()

    # Add target country info from document_id_2
    replications_with_country_1['target_country'] = replications_with_country_1['document_id_2'].map(project_country_lookup)

    # Drop rows where target country is missing or same as source
    replications_with_country_1 = replications_with_country_1.dropna(subset=['target_country'])
    replications_with_country_1 = replications_with_country_1[replications_with_country_1['target_country'] != country]

    # Get unique (source_doc, target_country) pairs
    all_replications = replications_with_country_1[['document_id_1', 'target_country']].drop_duplicates()

    # Keep only the FIRST replication per source document
    first_replications = all_replications.drop_duplicates(subset=['document_id_1'], keep='first')

    # Count number of unique documents replicated to each partner country
    partner_unique_docs = first_replications.groupby('target_country')['document_id_1'].nunique().reset_index()
    partner_unique_docs.columns = ['Replicated_Country', 'Unique_Replicated_Documents']

    # Total unique documents for the source country
    total_unique_docs = country_unique_replications.get(country, 1)  # default to 1 to avoid division by zero

    # Compute replication percentage
    partner_unique_docs['Replication_Percentage'] = (
        partner_unique_docs['Unique_Replicated_Documents'] / total_unique_docs * 100
    )

    # Top 3 replication partners
    top_3_partners = partner_unique_docs.sort_values(by='Replication_Percentage', ascending=False).head(3)

    top_country_partners[country] = top_3_partners

# Final print
for country, top_3_df in top_country_partners.items():
    print(f"\nTop 3 replication partners for {country} (by % of unique replications):")
    print(top_3_df)



Top 3 replication partners for Azerbaijan (by % of unique replications):
  Replicated_Country  Unique_Replicated_Documents  Replication_Percentage
0             AFRICA                            1                9.090909
1             Brazil                            1                9.090909
2              Congo                            1                9.090909

Top 3 replication partners for EUROPE AND CENTRAL ASIA (by % of unique replications):
  Replicated_Country  Unique_Replicated_Documents  Replication_Percentage
0            Albania                            1                    6.25
1         Bangladesh                            1                    6.25
2              Benin                            1                    6.25

Top 3 replication partners for Bosnia and Herzegovina (by % of unique replications):
  Replicated_Country  Unique_Replicated_Documents  Replication_Percentage
7         Tajikistan                            2               14.285714
1            

In [ ]:
country_coords = {
    "Afghanistan": (33.93911, 67.709953),
    "Albania": (41.3275, 20.8189),
    "Angola": (-11.2027, 17.8739),
    "Bangladesh": (23.684994, 90.356331),
    "Belarus": (53.7098, 27.9534),
    "Belize": (17.1899, -88.4976),
    "Benin": (9.3077, 2.3158),
    "Bolivia": (-16.2902, -63.5887),
    "Botswana": (-22.3285, 24.6849),
    "Brazil": (-14.235004, -51.92528),
    "Burkina Faso": (12.2383, -1.5616),
    "Burundi": (-3.3731, 29.9189),
    "Cabo Verde": (16.5388, -23.0418),
    "Cambodia": (12.5657, 104.991),
    "Central African Republic": (6.6111, 20.9394),
    "Chad": (15.4542, 18.7322),
    "Chile": (-35.6751, -71.543),
    "China": (35.8617, 104.1954),
    "Comoros": (-11.6455, 43.3333),
    "Congo": (-0.6606, 14.8966),
    "Costa Rica": (9.7489, -83.7534),
    "Croatia": (45.1, 15.2),
    "Djibouti": (11.8251, 42.5903),
    "Dominican Republic": (18.7357, -70.1627),
    "Ecuador": (-1.8312, -78.1834),
    "El Salvador": (13.7942, -88.8965),
    "Ethiopia": (9.145, 40.4897),
    "Georgia": (42.3154, 43.3569),
    "Haiti": (18.9712, -72.2852),
    "Honduras": (15.2, -86.2419),
    "Japan": (36.2048, 138.2529),
    "Kenya": (-0.0236, 37.9062),
    "Lao PDR": (19.8563, 102.4955),
    "Montenegro": (42.7087, 19.3744),
    "Pakistan": (30.3753, 69.3451),
    "Russia": (61.524, 105.3188),
    "Senegal": (14.4974, -14.4524),
    "Sri Lanka": (7.8731, 80.7718),
    "Tajikistan": (38.861034, 71.276093),
    "Uganda": (1.3733, 32.2903),
    "Uzbekistan": (41.3775, 64.5853),
    "Zambia": (-13.1339, 27.8493)
}


In [ ]:
replication_counts = pd.DataFrame([

    {"country_1": "Azerbaijan", "country_2": "Brazil", "percentage": 9},
    {"country_1": "Azerbaijan", "country_2": "Congo", "percentage": 9},


    {"country_1": "Bosnia and Herzegovina", "country_2": "Tajikistan", "percentage": 14},
    {"country_1": "Bosnia and Herzegovina", "country_2": "Albania", "percentage": 7},


    {"country_1": "Afghanistan", "country_2": "Cambodia", "percentage": 7},
    {"country_1": "Afghanistan", "country_2": "Dominican Republic", "percentage": 7},


    {"country_1": "Armenia", "country_2": "Honduras", "percentage": 18},
    {"country_1": "Armenia", "country_2": "El Salvador", "percentage": 9},


    {"country_1": "El Salvador", "country_2": "Bolivia", "percentage": 9},
    {"country_1": "El Salvador", "country_2": "Brazil", "percentage": 9},



    {"country_1": "Romania", "country_2": "Brazil", "percentage": 23},
    {"country_1": "Romania", "country_2": "Montenegro", "percentage": 15},


    {"country_1": "Sri Lanka", "country_2": "Chad", "percentage": 6},
    {"country_1": "Sri Lanka", "country_2": "Georgia", "percentage": 6},

    {"country_1": "Nicaragua", "country_2": "Croatia", "percentage": 8},
    {"country_1": "Nicaragua", "country_2": "Ecuador", "percentage": 8},


    {"country_1": "Zambia", "country_2": "Angola", "percentage": 6},
    {"country_1": "Zambia", "country_2": "Brazil", "percentage": 6},


    {"country_1": "Lebanon", "country_2": "Brazil", "percentage": 8},
    {"country_1": "Lebanon", "country_2": "Comoros", "percentage": 8},

    {"country_1": "Guatemala", "country_2": "Ecuador", "percentage": 25},
    {"country_1": "Guatemala", "country_2": "Belize", "percentage": 12.5},


    {"country_1": "Tajikistan", "country_2": "Costa Rica", "percentage": 10},
    {"country_1": "Tajikistan", "country_2": "Uzbekistan", "percentage": 10},


    {"country_1": "Russia", "country_2": "Belarus", "percentage": 11},
    {"country_1": "Russia", "country_2": "Croatia", "percentage": 11},


    {"country_1": "Ecuador", "country_2": "Brazil", "percentage": 27},
    {"country_1": "Ecuador", "country_2": "Bangladesh", "percentage": 7},

    {"country_1": "Chad", "country_2": "Brazil", "percentage": 8},
    {"country_1": "Chad", "country_2": "Chile", "percentage": 8},

])


In [ ]:
top_15_data = pd.DataFrame({
    'Country': [
        'Azerbaijan', 'Bosnia and Herzegovina',
        'Afghanistan', 'Armenia', 'El Salvador',
        'Romania', 'Sri Lanka', 'Nicaragua', 'Zambia',
        'Lebanon', 'Guatemala', 'Tajikistan', 'Russia', 'Ecuador',
        'Chad'
    ],
    'Replication_Percentage': [
        100.0, 93.333333,
        92.857143, 84.615385, 84.615385,
        81.25, 80.952381, 80.0, 80.0,
        80.0, 80.0, 77.777778, 75.0, 75.0,
        75.0
    ]
})


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Function to interpolate great circle midpoint
def interpolate_great_circle(lat1, lon1, lat2, lon2, fraction=0.5):
    lat1_rad, lon1_rad = np.radians([lat1, lon1])
    lat2_rad, lon2_rad = np.radians([lat2, lon2])

    delta = np.arccos(
        np.sin(lat1_rad) * np.sin(lat2_rad) +
        np.cos(lat1_rad) * np.cos(lat2_rad) * np.cos(lon2_rad - lon1_rad)
    )

    if delta == 0:
        return lat1, lon1

    A = np.sin((1 - fraction) * delta) / np.sin(delta)
    B = np.sin(fraction * delta) / np.sin(delta)

    x = A * np.cos(lat1_rad) * np.cos(lon1_rad) + B * np.cos(lat2_rad) * np.cos(lon2_rad)
    y = A * np.cos(lat1_rad) * np.sin(lon1_rad) + B * np.cos(lat2_rad) * np.sin(lon2_rad)
    z = A * np.sin(lat1_rad) + B * np.sin(lat2_rad)

    lat_mid = np.arctan2(z, np.sqrt(x**2 + y**2))
    lon_mid = np.arctan2(y, x)

    return np.degrees(lat_mid), np.degrees(lon_mid)

# Country coordinates (some countries are missing coordinates)



# Initialize the figure
fig = go.Figure()
# Set color scale boundaries
# First, create the full_fill_df and get top 15 countries
# Fix Congo name


top_15_data['Country'] = top_15_data['Country'].replace({
    'Congo': 'Congo, Rep.'
})

# Merge with all countries
all_country_names = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/2014_world_gdp_with_codes.csv")['COUNTRY'].unique()
full_fill_df = pd.DataFrame({'Country': all_country_names})

# Merge: Countries not in top 15 get Replication_Percentage = 0
full_fill_df = full_fill_df.merge(
    top_15_data[['Country', 'Replication_Percentage']],
    on='Country',
    how='left'
)
full_fill_df['Replication_Percentage'] = full_fill_df['Replication_Percentage'].fillna(0)

# Set color scale boundaries
zmin = 0
zmax = top_15_data['Replication_Percentage'].max()

# Create figure
fig = go.Figure()

fig.add_trace(go.Choropleth(
    locations=full_fill_df['Country'],
    locationmode='country names',
    z=full_fill_df['Replication_Percentage'],
    colorscale=[
        [0.0, '#f0f6fb'],   # Very light blue for non-top countries
        [0.3, '#d1e5f0'],   # Light-medium blue
        [0.6, '#74add1'],   # Medium blue
        [0.85, '#2b8cbe'],  # Darker blue
        [1.0, '#045a8d']    # Deep dark blue for highest
    ],
    zmin=zmin,
    zmax=zmax,
    colorbar=dict(title='Replication %', thickness=15, len=0.5),
    marker_line_color='white',
    showscale=True
))



# ➔ Now add country names (only for top 15 countries)
for _, row in top_15_data.iterrows():
    country = row['Country']
    replication_percentage = row['Replication_Percentage']
    coord = country_coords.get(country)
    if coord:
        lat, lon = coord
        fig.add_trace(go.Scattergeo(
            lon=[lon],
            lat=[lat],
            text=[f"{country} ({round(replication_percentage)}%)"],  # <<< Rounded to no decimals
            mode='text',
            showlegend=False,
            textfont=dict(
                size=12,
                color='black',
                family='Arial'
            )
        ))



# ➔ Update the layout to make full screen
fig.update_layout(
    title='Replication Percentage Across Top 15 Countries',
    geo=dict(
        projection_type='natural earth',
        showland=True,
        landcolor="#f0f6fb",
        showcountries=True,
        countrycolor='gray',
        showcoastlines=True,
        showframe=False
    ),
    height=750,
    margin={"r": 20, "t": 50, "l": 20, "b": 0}
)

# Loop over the connections and add traces
for _, row in replication_counts.iterrows():
    # Ensure both countries have valid coordinates
    if pd.isna(row['percentage']):
        continue  # Skip this row if percentage is missing

    # If the coordinates are missing, ensure they are set properly for each country
    lat1, lon1 = country_coords.get(row['country_1'], (None, None))
    lat2, lon2 = country_coords.get(row['country_2'], (None, None))

    if lat1 is None or lon1 is None or lat2 is None or lon2 is None:
        continue  # Skip if any coordinates are missing

    # Calculate the line thickness based on replication percentage
    line_thickness = 1 + 4 * row['percentage'] / 100  # Normalize percentage to fit desired line thickness range

    # Determine line color and style
    color = 'green' if row['country_1'] == row['country_2'] else 'darkblue'
    line_style = 'solid'

    needs_custom_path = (
        (row['country_1'] == 'China' and row['country_2'] == 'Brazil') or
        (row['country_1'] == 'Mexico' and row['country_2'] == 'China')
    )

    if needs_custom_path:
        mid_lat, mid_lon = interpolate_great_circle(lat1, lon1, lat2, lon2, 0.5)
        mid_lon = 0
        lats = [lat1, mid_lat, lat2]
        lons = [lon1, mid_lon, lon2]
    else:
        lats = [lat1, lat2]
        lons = [lon1, lon2]

    # Add trace for the connection
    fig.add_trace(go.Scattergeo(
        lon=lons,
        lat=lats,
        mode='lines',
        line=dict(width=line_thickness, color=color, dash=line_style),
        opacity=0.8,
        hoverinfo='text',
        text=f"{row['country_1']} → {row['country_2']} ({row['percentage']}% replication)",
        showlegend=False
    ))

    # Add a label at the midpoint of the line for context
    mid_lat_label, mid_lon_label = interpolate_great_circle(lat1, lon1, lat2, lon2, 0.5)
    fig.add_trace(go.Scattergeo(
        lon=[mid_lon_label],
        lat=[mid_lat_label],
        mode='text',
        text=[f"{row['percentage']}%"],
        showlegend=False,
        textfont=dict(size=10, color='black', family="Arial"),
        opacity=0.8,
        hoverinfo='none',
    ))


fig.update_layout(
    title='Replication Percentage Across Top 15 Countries',
    geo=dict(
        projection_type='natural earth',
        showland=True,
        landcolor="#f0f6fb",
        showcountries=True,
        countrycolor='gray',
        showcoastlines=True,
        showframe=False
    ),
    height=750,
    margin={"r": 20, "t": 50, "l": 20, "b": 0}
)
# Show the figure
fig.show()
